# 02 Clean And Prepare States

Clean Frankfurt Bronze state vectors, convert them into NM-based horizontal cells and 3000-ft vertical cells, and write the V2 Silver analysis base.

Primary targets:

- `adsb_airspace_eddf.slv_airspace.flight_states_cellized_v2`
- `adsb_airspace_eddf.ref.cell_schemes_v2`
- `adsb_airspace_eddf.ref.airspace_cells_v2`
- `adsb_airspace_eddf.obs.pipeline_run_log`


## V2 Cellization Contract

This notebook keeps the Bronze source unchanged and writes a V2 Silver layer for sensitivity analysis.

- Horizontal discretization: `horizontal_cell_nm x horizontal_cell_nm`
- Vertical discretization: `vertical_cell_ft`
- Time discretization: `analysis_window_minutes`
- Cell identity: `horizontal_cell_id + vertical_cell_id`
- Scheme identity: `cell_scheme_id`


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from math import cos, radians
from pathlib import Path
import json
import sys
import yaml

from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"

def parse_float(raw_value: str) -> float:
    return float(raw_value.strip())

def parse_int(raw_value: str) -> int:
    return int(raw_value.strip())

def parse_optional_int(raw_value: str):
    text = raw_value.strip()
    return None if not text else int(text)

def utc_now_naive() -> datetime:
    return datetime.now(UTC).replace(tzinfo=None)

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value

def append_row_to_table(*, target_table: str, payload: dict) -> None:
    target_schema = spark.table(target_table).schema
    row = {field.name: normalize_scalar(payload.get(field.name)) for field in target_schema}
    spark.createDataFrame([row], schema=target_schema).write.mode("append").saveAsTable(target_table)

def format_for_id(value) -> str:
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return str(numeric).replace('.', 'p')

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_source_hour_epoch = ""
default_horizontal_cell_nm = "10"
default_vertical_cell_ft = "3000"
default_analysis_window_minutes = "15"
default_min_altitude_ft = "0"
default_max_altitude_ft = "24500"
default_projection_mode = "local_nm"
default_include_on_ground = "false"
default_overwrite_source_run = "true"

catalog = default_catalog
source_run_id_raw = default_source_run_id
source_hour_epoch_raw = default_source_hour_epoch
horizontal_cell_nm_raw = default_horizontal_cell_nm
vertical_cell_ft_raw = default_vertical_cell_ft
analysis_window_minutes_raw = default_analysis_window_minutes
min_altitude_ft_raw = default_min_altitude_ft
max_altitude_ft_raw = default_max_altitude_ft
projection_mode = default_projection_mode
include_on_ground_raw = default_include_on_ground
overwrite_source_run_raw = default_overwrite_source_run

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.text("source_hour_epoch", default_source_hour_epoch)
    dbutils.widgets.text("horizontal_cell_nm", default_horizontal_cell_nm)
    dbutils.widgets.text("vertical_cell_ft", default_vertical_cell_ft)
    dbutils.widgets.text("analysis_window_minutes", default_analysis_window_minutes)
    dbutils.widgets.text("min_altitude_ft", default_min_altitude_ft)
    dbutils.widgets.text("max_altitude_ft", default_max_altitude_ft)
    dbutils.widgets.dropdown("projection_mode", default_projection_mode, ["local_nm"])
    dbutils.widgets.dropdown("include_on_ground", default_include_on_ground, ["true", "false"])
    dbutils.widgets.dropdown("overwrite_source_run", default_overwrite_source_run, ["true", "false"])

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    source_hour_epoch_raw = dbutils.widgets.get("source_hour_epoch").strip()
    horizontal_cell_nm_raw = dbutils.widgets.get("horizontal_cell_nm").strip() or default_horizontal_cell_nm
    vertical_cell_ft_raw = dbutils.widgets.get("vertical_cell_ft").strip() or default_vertical_cell_ft
    analysis_window_minutes_raw = dbutils.widgets.get("analysis_window_minutes").strip() or default_analysis_window_minutes
    min_altitude_ft_raw = dbutils.widgets.get("min_altitude_ft").strip() or default_min_altitude_ft
    max_altitude_ft_raw = dbutils.widgets.get("max_altitude_ft").strip() or default_max_altitude_ft
    projection_mode = dbutils.widgets.get("projection_mode").strip() or default_projection_mode
    include_on_ground_raw = dbutils.widgets.get("include_on_ground")
    overwrite_source_run_raw = dbutils.widgets.get("overwrite_source_run")

focus_airport = region_config["focus_airport"]
bbox = region_config["regional_bbox"]
bbox_min_lat = float(bbox["min_latitude"])
bbox_max_lat = float(bbox["max_latitude"])
bbox_min_lon = float(bbox["min_longitude"])
bbox_max_lon = float(bbox["max_longitude"])
horizontal_cell_nm = parse_float(horizontal_cell_nm_raw)
vertical_cell_ft = parse_float(vertical_cell_ft_raw)
analysis_window_minutes = parse_int(analysis_window_minutes_raw)
min_altitude_ft = parse_float(min_altitude_ft_raw)
max_altitude_ft = parse_float(max_altitude_ft_raw)
include_on_ground = parse_bool(include_on_ground_raw)
overwrite_source_run = parse_bool(overwrite_source_run_raw)
source_hour_epoch = parse_optional_int(source_hour_epoch_raw)
source_run_id_input = source_run_id_raw.strip()

if projection_mode != "local_nm":
    raise ValueError("Only projection_mode='local_nm' is supported in V2 for now.")
if horizontal_cell_nm <= 0 or vertical_cell_ft <= 0 or analysis_window_minutes <= 0:
    raise ValueError("horizontal_cell_nm, vertical_cell_ft, and analysis_window_minutes must be positive")
if min_altitude_ft < 0 or max_altitude_ft <= min_altitude_ft:
    raise ValueError("Altitude window must satisfy 0 <= min_altitude_ft < max_altitude_ft")
if source_hour_epoch is not None and overwrite_source_run:
    raise ValueError(
        "Refusing to overwrite an entire source run when source_hour_epoch is set. Clear source_hour_epoch or set overwrite_source_run=false."
    )

bronze_table = f"{catalog}.brz_adsb.hist_state_vectors"
silver_table_v2 = f"{catalog}.slv_airspace.flight_states_cellized_v2"
cell_schemes_table_v2 = f"{catalog}.ref.cell_schemes_v2"
airspace_cells_table_v2 = f"{catalog}.ref.airspace_cells_v2"
pipeline_log_table = f"{catalog}.obs.pipeline_run_log"

cell_scheme_id = (
    f"{focus_airport.lower()}_h{format_for_id(horizontal_cell_nm)}nm_"
    f"v{format_for_id(vertical_cell_ft)}ft_"
    f"w{analysis_window_minutes}m_"
    f"a{format_for_id(min_altitude_ft)}to{format_for_id(max_altitude_ft)}_"
    f"{projection_mode}_v2"
)


In [ ]:
if source_run_id_input:
    selected_source_run_id = source_run_id_input
else:
    latest_success_row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.ingestion_log
        WHERE source_object = 'state_vectors_data4'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if latest_success_row is None:
        raise ValueError("No successful state_vectors_data4 Bronze run found. Run 01a_ingest_opensky_history.ipynb first.")
    selected_source_run_id = latest_success_row["run_id"]

bronze_df = spark.table(bronze_table).where(F.col("run_id") == F.lit(selected_source_run_id))
if source_hour_epoch is not None:
    bronze_df = bronze_df.where(F.col("hour") == F.lit(source_hour_epoch))

source_row_count = bronze_df.count()
if source_row_count == 0:
    raise ValueError(f"No Bronze rows found for run_id={selected_source_run_id} and source_hour_epoch={source_hour_epoch}.")

source_summary_row = bronze_df.agg(
    F.countDistinct("hour").alias("hour_partition_count"),
    F.countDistinct("icao24").alias("aircraft_count"),
    F.min("hour").alias("min_hour"),
    F.max("hour").alias("max_hour")
).first()

run_plan = {
    "catalog": catalog,
    "source_run_id": selected_source_run_id,
    "source_hour_epoch": source_hour_epoch,
    "bronze_table": bronze_table,
    "silver_table_v2": silver_table_v2,
    "cell_schemes_table_v2": cell_schemes_table_v2,
    "airspace_cells_table_v2": airspace_cells_table_v2,
    "horizontal_cell_nm": horizontal_cell_nm,
    "vertical_cell_ft": vertical_cell_ft,
    "analysis_window_minutes": analysis_window_minutes,
    "min_altitude_ft": min_altitude_ft,
    "max_altitude_ft": max_altitude_ft,
    "projection_mode": projection_mode,
    "cell_scheme_id": cell_scheme_id,
    "include_on_ground": include_on_ground,
    "overwrite_source_run": overwrite_source_run,
    "source_row_count": source_row_count,
    "source_aircraft_count": int(source_summary_row["aircraft_count"]),
    "source_hour_partition_count": int(source_summary_row["hour_partition_count"]),
    "min_hour": int(source_summary_row["min_hour"]),
    "max_hour": int(source_summary_row["max_hour"]),
}

run_plan


In [ ]:
meters_to_feet = 3.28083989501312
meters_per_second_to_knots = 1.9438444924406
meters_per_second_to_fpm = 196.850393700787
coordinate_epsilon = 1e-9
altitude_epsilon = 1e-6
latitude_origin = bbox_min_lat
longitude_origin = bbox_min_lon
cos_origin = cos(radians(latitude_origin))
if abs(cos_origin) < 1e-9:
    raise ValueError("local_nm projection is unstable at the selected origin latitude")
lat_step_deg = horizontal_cell_nm / 60.0
lon_step_deg = horizontal_cell_nm / (60.0 * cos_origin)
analysis_window_seconds = analysis_window_minutes * 60

working_df = (
    bronze_df
    .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("time"))))
    .withColumn(
        "analysis_window_start",
        F.to_timestamp(F.from_unixtime(F.floor(F.col("time") / F.lit(analysis_window_seconds)) * F.lit(analysis_window_seconds)))
    )
    .withColumn("callsign_clean", F.when(F.length(F.trim(F.col("callsign"))) > 0, F.trim(F.col("callsign"))))
    .withColumn("latitude", F.col("lat").cast("double"))
    .withColumn("longitude", F.col("lon").cast("double"))
    .withColumn("altitude_ft_raw", F.col("baroaltitude") * F.lit(meters_to_feet))
    .withColumn(
        "altitude_ft",
        F.when(F.col("baroaltitude").isNull(), F.lit(None)).otherwise(F.greatest(F.col("altitude_ft_raw"), F.lit(0.0)))
    )
    .withColumn("speed_kt", F.when(F.col("velocity").isNull(), F.lit(None)).otherwise(F.col("velocity") * F.lit(meters_per_second_to_knots)))
    .withColumn("heading_deg", F.when((F.col("heading") >= 0) & (F.col("heading") <= 360), F.col("heading")).otherwise(F.lit(None)))
    .withColumn("vertical_rate_fpm", F.when(F.col("vertrate").isNull(), F.lit(None)).otherwise(F.col("vertrate") * F.lit(meters_per_second_to_fpm)))
    .where(F.col("icao24").isNotNull())
    .where(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .where(F.col("latitude").between(bbox_min_lat, bbox_max_lat))
    .where(F.col("longitude").between(bbox_min_lon, bbox_max_lon))
    .where(F.col("baroaltitude").isNotNull())
    .where((F.col("velocity").isNull()) | ((F.col("velocity") >= 0) & (F.col("velocity") <= 350)))
)

if not include_on_ground:
    working_df = working_df.where(~F.coalesce(F.col("onground"), F.lit(False)))

working_df = (
    working_df
    .where(F.col("altitude_ft").between(F.lit(min_altitude_ft), F.lit(max_altitude_ft)))
    .withColumn("latitude_for_cell", F.when(F.col("latitude") >= F.lit(bbox_max_lat), F.lit(bbox_max_lat - coordinate_epsilon)).otherwise(F.col("latitude")))
    .withColumn("longitude_for_cell", F.when(F.col("longitude") >= F.lit(bbox_max_lon), F.lit(bbox_max_lon - coordinate_epsilon)).otherwise(F.col("longitude")))
    .withColumn("altitude_for_cell", F.when(F.col("altitude_ft") >= F.lit(max_altitude_ft), F.lit(max_altitude_ft - altitude_epsilon)).otherwise(F.col("altitude_ft")))
    .withColumn("north_nm", (F.col("latitude_for_cell") - F.lit(latitude_origin)) * F.lit(60.0))
    .withColumn("east_nm", (F.col("longitude_for_cell") - F.lit(longitude_origin)) * F.lit(60.0 * cos_origin))
    .withColumn("horizontal_index_y", F.floor(F.col("north_nm") / F.lit(horizontal_cell_nm)).cast("int"))
    .withColumn("horizontal_index_x", F.floor(F.col("east_nm") / F.lit(horizontal_cell_nm)).cast("int"))
    .withColumn("vertical_index", F.floor((F.col("altitude_for_cell") - F.lit(min_altitude_ft)) / F.lit(vertical_cell_ft)).cast("int"))
    .withColumn("horizontal_cell_id", F.format_string("fra_h_%03d_%03d", F.col("horizontal_index_y"), F.col("horizontal_index_x")))
    .withColumn("vertical_cell_id", F.format_string("z_%03d", F.col("vertical_index")))
    .withColumn("cell_id", F.concat_ws("_", F.col("horizontal_cell_id"), F.col("vertical_cell_id")))
)

silver_df_v2 = working_df.select(
    F.col("event_time"),
    F.col("analysis_window_start"),
    F.col("horizontal_cell_id"),
    F.col("vertical_cell_id"),
    F.col("cell_id"),
    F.col("horizontal_index_x"),
    F.col("horizontal_index_y"),
    F.col("vertical_index"),
    F.col("icao24"),
    F.col("callsign_clean").alias("callsign"),
    F.col("latitude"),
    F.col("longitude"),
    F.round(F.col("altitude_ft"), 2).alias("altitude_ft"),
    F.round(F.col("speed_kt"), 2).alias("speed_kt"),
    F.round(F.col("heading_deg"), 4).alias("heading_deg"),
    F.round(F.col("vertical_rate_fpm"), 2).alias("vertical_rate_fpm"),
    F.lit(focus_airport).alias("focus_airport"),
    F.lit(bronze_table).alias("source_table"),
    F.lit(cell_scheme_id).alias("cell_scheme_id"),
    F.current_timestamp().alias("ingested_at"),
    F.lit(selected_source_run_id).alias("run_id"),
)

airspace_cells_df_v2 = (
    working_df
    .select("cell_id", "horizontal_cell_id", "vertical_cell_id", "horizontal_index_x", "horizontal_index_y", "vertical_index")
    .distinct()
    .withColumn("min_latitude", F.lit(latitude_origin) + F.col("horizontal_index_y") * F.lit(lat_step_deg))
    .withColumn("max_latitude", F.col("min_latitude") + F.lit(lat_step_deg))
    .withColumn("min_longitude", F.lit(longitude_origin) + F.col("horizontal_index_x") * F.lit(lon_step_deg))
    .withColumn("max_longitude", F.col("min_longitude") + F.lit(lon_step_deg))
    .withColumn("center_latitude", (F.col("min_latitude") + F.col("max_latitude")) / F.lit(2.0))
    .withColumn("center_longitude", (F.col("min_longitude") + F.col("max_longitude")) / F.lit(2.0))
    .withColumn("min_altitude_ft", F.lit(min_altitude_ft) + F.col("vertical_index") * F.lit(vertical_cell_ft))
    .withColumn("max_altitude_ft", F.least(F.col("min_altitude_ft") + F.lit(vertical_cell_ft), F.lit(max_altitude_ft)))
    .withColumn("horizontal_cell_nm", F.lit(horizontal_cell_nm))
    .withColumn("vertical_cell_ft", F.lit(vertical_cell_ft))
    .withColumn("projection_mode", F.lit(projection_mode))
    .withColumn("focus_airport", F.lit(focus_airport))
    .withColumn("active_flag", F.lit(True))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("cell_scheme_id", F.lit(cell_scheme_id))
    .withColumn("run_id", F.lit(selected_source_run_id))
    .select(
        "cell_id",
        "horizontal_cell_id",
        "vertical_cell_id",
        "horizontal_index_x",
        "horizontal_index_y",
        "vertical_index",
        "min_latitude",
        "max_latitude",
        "min_longitude",
        "max_longitude",
        "center_latitude",
        "center_longitude",
        "min_altitude_ft",
        "max_altitude_ft",
        "horizontal_cell_nm",
        "vertical_cell_ft",
        "projection_mode",
        "focus_airport",
        "active_flag",
        "created_at",
        "cell_scheme_id",
        "run_id",
    )
)

quality_row = silver_df_v2.agg(
    F.count("*").alias("cellized_row_count"),
    F.countDistinct("icao24").alias("aircraft_count"),
    F.countDistinct("horizontal_cell_id").alias("horizontal_cell_count"),
    F.countDistinct("vertical_cell_id").alias("vertical_cell_count"),
    F.countDistinct("cell_id").alias("cell_count"),
    F.min("event_time").alias("min_event_time"),
    F.max("event_time").alias("max_event_time")
).first()

clean_summary = {
    "source_run_id": selected_source_run_id,
    "source_hour_epoch": source_hour_epoch,
    "cell_scheme_id": cell_scheme_id,
    "source_row_count": source_row_count,
    "cellized_row_count": int(quality_row["cellized_row_count"]),
    "aircraft_count": int(quality_row["aircraft_count"]),
    "horizontal_cell_count": int(quality_row["horizontal_cell_count"]),
    "vertical_cell_count": int(quality_row["vertical_cell_count"]),
    "cell_count": int(quality_row["cell_count"]),
    "min_event_time": str(quality_row["min_event_time"]),
    "max_event_time": str(quality_row["max_event_time"]),
}

if clean_summary["cellized_row_count"] == 0:
    raise ValueError("Cellization removed all rows. Relax the filters or inspect the Bronze source window.")


In [ ]:
pipeline_started_at = utc_now_naive()
pipeline_status = "success"
pipeline_error_message = None

scheme_payload = {
    "cell_scheme_id": cell_scheme_id,
    "focus_airport": focus_airport,
    "horizontal_cell_nm": horizontal_cell_nm,
    "vertical_cell_ft": vertical_cell_ft,
    "analysis_window_minutes": analysis_window_minutes,
    "min_altitude_ft": min_altitude_ft,
    "max_altitude_ft": max_altitude_ft,
    "projection_mode": projection_mode,
    "minimum_active_windows": 3,
    "created_at": utc_now_naive(),
    "run_id": selected_source_run_id,
}

try:
    if overwrite_source_run:
        spark.sql(f"DELETE FROM {silver_table_v2} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{cell_scheme_id}'")
        spark.sql(f"DELETE FROM {airspace_cells_table_v2} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{cell_scheme_id}'")
        spark.sql(f"DELETE FROM {cell_schemes_table_v2} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{cell_scheme_id}'")

    append_row_to_table(target_table=cell_schemes_table_v2, payload=scheme_payload)
    airspace_cells_df_v2.write.mode("append").saveAsTable(airspace_cells_table_v2)
    silver_df_v2.write.mode("append").saveAsTable(silver_table_v2)
except Exception as exc:
    pipeline_status = "failed"
    pipeline_error_message = f"{type(exc).__name__}: {exc}"
    raise
finally:
    append_row_to_table(
        target_table=pipeline_log_table,
        payload={
            "run_id": selected_source_run_id,
            "pipeline_name": "02_clean_and_prepare_states",
            "layer": "silver_v2",
            "target_table": silver_table_v2,
            "status": pipeline_status,
            "rows_read": source_row_count,
            "rows_written": clean_summary["cellized_row_count"] if pipeline_status == "success" else 0,
            "started_at": pipeline_started_at,
            "completed_at": utc_now_naive(),
            "error_message": pipeline_error_message,
            "metadata_json": json.dumps(
                {
                    "cell_scheme_id": cell_scheme_id,
                    "horizontal_cell_nm": horizontal_cell_nm,
                    "vertical_cell_ft": vertical_cell_ft,
                    "analysis_window_minutes": analysis_window_minutes,
                    "min_altitude_ft": min_altitude_ft,
                    "max_altitude_ft": max_altitude_ft,
                    "projection_mode": projection_mode,
                    "include_on_ground": include_on_ground,
                    "source_hour_epoch": source_hour_epoch,
                },
                sort_keys=True,
                default=str,
            ),
        },
    )

final_summary = {
    "status": pipeline_status,
    "source_run_id": selected_source_run_id,
    "source_hour_epoch": source_hour_epoch,
    "cell_scheme_id": cell_scheme_id,
    "silver_table_v2": silver_table_v2,
    "airspace_cells_table_v2": airspace_cells_table_v2,
    "source_row_count": source_row_count,
    "cellized_row_count": clean_summary["cellized_row_count"],
    "aircraft_count": clean_summary["aircraft_count"],
    "horizontal_cell_count": clean_summary["horizontal_cell_count"],
    "vertical_cell_count": clean_summary["vertical_cell_count"],
    "cell_count": clean_summary["cell_count"],
}

final_summary
